# Day 2 Data Collection and Cleaning

This notebook is the minimal Day 2 deliverable for the UK cost-of-living / inequality project.
The objective is to show how the raw datasets were collected, cleaned, validated, and saved into
tidy processed files that can support later analysis.

Each run is intended to fetch the latest published official data. If a required source cannot be
refreshed, the pipeline should stop rather than silently use an older local raw file.

## Key Day 2 Decisions

- Inflation uses the ONS CPIH annual rate because it is a broad UK inflation measure and includes owner occupiers' housing costs.
- Housing is represented by the UK House Price Index for the United Kingdom.
- Geography is standardized to `UK` in all processed outputs.
- Annual income remains annual and is **not** expanded to monthly frequency.
- If `year_month` appears in the income file, it is only a reference anchor for the annual period end.
- Structural missingness is reported rather than auto-imputed.
- "Latest data" here means the latest **published official release**, not a real-time feed.
- Raw filenames use the fixed `20260414` Day 2 submission snapshot stamp for consistency.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_files = {
    "Inflation": project_root / "data/raw/ons_inflation_20260414.csv",
    "Housing": project_root / "data/raw/ons_house_prices_20260414.csv",
    "Income": project_root / "data/raw/ons_income_decile_20260414.xlsx",
    "Bank Rate": project_root / "data/raw/boe_bank_rate_20260414.csv",
    "Unemployment": project_root / "data/raw/ons_unemployment_20260414.csv",
}

processed_files = {
    "Inflation": project_root / "data/processed/day2_inflation_clean.csv",
    "Housing": project_root / "data/processed/day2_house_prices_clean.csv",
    "Bank Rate": project_root / "data/processed/day2_bank_rate_clean.csv",
    "Unemployment": project_root / "data/processed/day2_unemployment_clean.csv",
    "Monthly Macro": project_root / "data/processed/day2_merged_monthly_macro.csv",
    "Income": project_root / "data/processed/day2_income_decile_clean.csv",
}

print("Raw files")
for label, path in raw_files.items():
    print(f"  - {label}: {path}")

print("\nProcessed files")
for label, path in processed_files.items():
    print(f"  - {label}: {path}")

## Preview Raw Data

A small preview is enough for Day 2. The goal is to confirm that the raw files exist and look like
the expected official downloads before cleaning.

In [ ]:
raw_preview_specs = {
    "Inflation": {"kind": "csv", "kwargs": {"header": None, "nrows": 5}},
    "Housing": {"kind": "csv", "kwargs": {"nrows": 5}},
    "Income": {"kind": "excel", "kwargs": {"nrows": 5}},
    "Bank Rate": {"kind": "csv", "kwargs": {"nrows": 5}},
    "Unemployment": {"kind": "csv", "kwargs": {"header": None, "nrows": 5}},
}

for label, path in raw_files.items():
    print(f"\n=== {label} raw preview ===")
    spec = raw_preview_specs[label]
    if spec["kind"] == "excel":
        df = pd.read_excel(path, nrows=spec["kwargs"]["nrows"])
    else:
        df = pd.read_csv(path, **spec["kwargs"])
    display(df.head())

## Preview Cleaned Outputs

These are the final processed Day 2 tables saved by the pipeline.

In [ ]:
cleaned = {label: pd.read_csv(path) for label, path in processed_files.items()}

for label, df in cleaned.items():
    print(f"\n=== {label} cleaned data ===")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())

## Quick Integrity Checks

Day 2 should make the main QA checks visible:
- row counts
- date coverage
- duplicate-key checks
- missing-value counts

In [ ]:
def coverage_text(df, date_column="year_month"):
    if df.empty:
        return "no rows"
    return f"{df[date_column].min()} to {df[date_column].max()}"

checks = []
checks.append({
    "dataset": "Monthly Macro",
    "rows": len(cleaned["Monthly Macro"]),
    "coverage": coverage_text(cleaned["Monthly Macro"]),
    "duplicate_key_rows": int(cleaned["Monthly Macro"].duplicated(subset=["year_month"]).sum()),
    "missing_values": cleaned["Monthly Macro"].isna().sum().to_dict(),
})
checks.append({
    "dataset": "Income",
    "rows": len(cleaned["Income"]),
    "coverage": coverage_text(cleaned["Income"]),
    "duplicate_key_rows": int(cleaned["Income"].duplicated(subset=["year_month", "decile"]).sum()),
    "missing_values": cleaned["Income"].isna().sum().to_dict(),
})

checks_df = pd.DataFrame(checks)
display(checks_df)

## Merge And Frequency Notes

- The final monthly macro file uses a monthly `year_month` key.
- Inflation and housing define the shared monthly merge window.
- Bank Rate and unemployment are attached as monthly context variables.
- Annual income is kept in a separate file and is not mechanically merged into the monthly macro dataset.
- No structural missingness is auto-imputed.

In [ ]:
print("Final Day 2 processed outputs:")
for label, path in processed_files.items():
    print(f"  - {label}: {path}")